# 02 · Finetune recovery and its negative controls: is the gain real transfer?

**What is being tested.** The PARNET demo's head-finetune (a fresh K-task head - target + control +
additive mix + mix-coeff penalty - trained with MultinomialNLL on frozen body features), compared to the
pretrained PARNET, under three settings that share ONE dataset:
1. **main** - head on the frozen **PARNET body**, depth-normalized demo loss;
2. **Control 3 (random body)** - same head/training on a frozen **randomly-initialized** body;
3. **Control 2 (faithful objective)** - head on the PARNET body with the **faithful RBPNet objective**
 (raw-count MultinomialNLL + `additive_mix_max` + log2 task-balancing + min-height masking).

**Why test it.** A finetune that raises Pearson could be (a) real transfer from PARNET's representation,
(b) the head merely fitting peak-centered windows (a random body would too), or (c) an artifact of the
simplified normalized loss (the faithful objective would remove it). The two negative controls separate these.

**Reasoning.** If the random body recovers nearly as well, the gain is *not* transfer. If it vanishes under
the faithful objective, it is a *loss artifact*. To count as real, the gain must survive both. We build the
multi-track dataset **once** (the remote eCLIP reads are the cost) and train all three heads on it.

## Definitions: the profile loss and the bias mixture

For one window/track with observed counts $o\in\mathbb{Z}_{\ge 0}^{L}$ and total $N=\sum_i o_i$, the
profile $\hat p=\operatorname{softmax}(z)$ is trained by the **Multinomial negative log-likelihood**
$$\operatorname{NLL}=-\sum_{i=1}^{L} o_i\,\log \hat p_i \quad(\text{up to a constant}).$$
eCLIP is a mixture of a protein-specific **target** and an SMInput **control**; PARNET's `AdditiveMix`
head combines them with a learned per-track **mix-coefficient** $\pi\in(0,1)$,
$$p_{\mathrm{tot}}=\pi\,p_{\mathrm{tgt}}+(1-\pi)\,p_{\mathrm{ctrl}}\quad(\text{probability space}).$$
The **faithful RBPNet objective** (Control 2; Horlacher 2023) mixes in log space via
$\operatorname{additive\_mix\_max}$, uses raw-count (not depth-normalized) NLL, $\log_2$ task balancing
$w=1/\log_2(2+N)$, and min-height masking $N\ge N_{\min}$. We report recovery as the change in mean
Pearson, $\Delta=r_{\text{finetuned}}-r_{\text{pretrained}}$, read beside the random-body (Control 3) and
faithful-objective (Control 2) deltas.

In [1]:
import os, sys, json, pathlib
import numpy as np
_here = pathlib.Path.cwd().resolve()
REPO = next((c for c in (_here, *_here.parents) if (c / "src" / "mmpartnet").is_dir()), _here)
sys.path.insert(0, str(REPO / "src"))
from mmpartnet import config
print("repo:", REPO)

repo: /workspace/mmpartnet


In [2]:
from mmpartnet.io import groups
from mmpartnet.models.parnet import load_parnet
from mmpartnet.experiments import recover_demo_finetune as ft

# the finetune panel is kept small on purpose: build_dataset reads every track for every window
# (cost ~ windows x tracks), so a 5-RBP panel keeps the single remote-read pass tractable on a node.
GROUP  = os.environ.get("MMP_FT_GROUP", "SF3B4,U2AF2,PRPF8,AQR,RBM22")
CELL   = os.environ.get("MMP_CELL", "HepG2")
NWIN   = int(os.environ.get("MMP_FT_NWIN", "12"))
EPOCHS = int(os.environ.get("MMP_EPOCHS", "8"))
manifest = json.loads((config.DATA / "eclip_manifest.json").read_text())
rbps0 = [g for g in (groups.resolve(GROUP) or [GROUP]) if g in manifest]
print(f"finetune panel: group={GROUP} cell={CELL} nwin={NWIN} epochs={EPOCHS} RBPs={rbps0}")

m = load_parnet()
rbps, tracks, exps, F, OE, OC, seqs = ft.build_dataset(m, rbps0, CELL, manifest, NWIN)  # ONE remote-read pass
print(f"assembled {len(F)} windows x {len(rbps)} tracks (eCLIP + control)")
assert len(F) >= 12, "too few windows; widen the panel or raise MMP_FT_NWIN"
F_rand = ft.random_body_feats(seqs, m)   # Control-3 features from the SAME windows (no re-reads)
print("random-body features:", F_rand.shape)

finetune panel: group=SF3B4,U2AF2,PRPF8,AQR,RBM22 cell=HepG2 nwin=12 epochs=8 RBPs=['SF3B4', 'U2AF2', 'PRPF8', 'AQR', 'RBM22']


/workspace/envs/mmpartnet/lib/python3.12/site-packages/torch/nn/modules/conv.py:380: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1090.)
  return F.conv1d(


assembled 60 windows x 5 tracks (eCLIP + control)
random-body features: (60, 512, 600)


In [3]:
def means(rows):
    ev = [r for r in rows if r.get("n")]
    pp = float(np.mean([r["pretrained_pearson"] for r in ev])) if ev else float("nan")
    fp = float(np.mean([r["finetuned_pearson"] for r in ev])) if ev else float("nan")
    return pp, fp, sum(r.get("n", 0) for r in ev)

res = {}
res["main"]     = ft.train_eval_head(F, OE, OC, seqs, rbps, tracks, m, objective="normalized", epochs=EPOCHS)
res["randbody"] = ft.train_eval_head(F_rand, OE, OC, seqs, rbps, tracks, m, objective="normalized", epochs=EPOCHS)
res["rbpnet"]   = ft.train_eval_head(F, OE, OC, seqs, rbps, tracks, m, objective="rbpnet", epochs=EPOCHS)
M = {k: means(v) for k, v in res.items()}
for k in ("main", "randbody", "rbpnet"):
    pp, fp, n = M[k]
    print(f"  {k:9} pretrained={pp:+.3f}  finetuned={fp:+.3f}  delta={fp-pp:+.3f}  (N={n})")

  main      pretrained=+0.206  finetuned=+0.295  delta=+0.089  (N=85)
  randbody  pretrained=+0.206  finetuned=+0.163  delta=-0.043  (N=85)
  rbpnet    pretrained=+0.206  finetuned=+0.218  delta=+0.012  (N=85)


In [4]:
from IPython.display import Markdown, display
def dl(k): pp, fp, _ = M[k]; return fp - pp
labels = {"main": "main (PARNET body, normalized)", "randbody": "Control 3 (random body)",
          "rbpnet": "Control 2 (faithful RBPNet obj)"}
md = "## Conclusion\n\n| setting | pretrained | finetuned | delta |\n|---|---|---|---|\n"
for k in ("main", "randbody", "rbpnet"):
    pp, fp, _ = M[k]
    md += f"| {labels[k]} | {pp:+.3f} | {fp:+.3f} | **{fp-pp:+.3f}** |\n"
v = []
v.append(f"transfer: PARNET-body delta {dl('main'):+.3f} vs random-body delta {dl('randbody'):+.3f} -> "
         + ("real transfer (PARNET body >> random body)" if dl('main') - dl('randbody') > 0.05
            else "random body explains the gain -- NOT transfer"))
v.append(f"objective: faithful-RBPNet delta {dl('rbpnet'):+.3f} -> "
         + ("gain survives the established objective" if dl('rbpnet') > 0.02
            else "gain weakens under the faithful objective (treat as a loss-sensitive effect)"))
display(Markdown(md + "\n" + "\n".join(f"- {s}" for s in v) +
    "\n\nThe finetune is reported only beside these two negative controls, so the headline delta is "
    "attributable to PARNET's learned representation under the established objective, not to peak-centring "
    "or a simplified loss. Claude-assisted, faithful to Horlacher 2023 (RBPNet AdditiveMix)."))

## Conclusion

| setting | pretrained | finetuned | delta |
|---|---|---|---|
| main (PARNET body, normalized) | +0.206 | +0.295 | **+0.089** |
| Control 3 (random body) | +0.206 | +0.163 | **-0.043** |
| Control 2 (faithful RBPNet obj) | +0.206 | +0.218 | **+0.012** |

- transfer: PARNET-body delta +0.089 vs random-body delta -0.043 -> real transfer (PARNET body >> random body)
- objective: faithful-RBPNet delta +0.012 -> gain weakens under the faithful objective (treat as a loss-sensitive effect)

The finetune is reported only beside these two negative controls, so the headline delta is attributable to PARNET's learned representation under the established objective, not to peak-centring or a simplified loss. Claude-assisted, faithful to Horlacher 2023 (RBPNet AdditiveMix).